<a href="https://colab.research.google.com/github/jiyeonlee-2930/Tourist-Prediction-Project-2026/blob/main/01_%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv("서울시설공단_어린이대공원 입장객 인원 정보_20260714.csv", encoding="utf-8-sig")

# 2. 날짜 타입 변환
df['입장일'] = pd.to_datetime(df['입장일'])

# 3. 지정된 기간 필터링 (2023-02-01 ~ 2025-01-31)
mask = (df['입장일'] >= '2023-02-01') & (df['입장일'] <= '2025-01-31')
filtered_df = df.loc[mask]

# 4. 일별 그룹화 및 합산
daily_df = filtered_df.groupby('입장일', as_index=False)['입장인원'].sum()
daily_df.rename(columns={'입장인원': '총입장객수'}, inplace=True)

# 5. 결과 확인
print(daily_df.head())
print(f"총 데이터 개수: {len(daily_df)}일")

# 6. 전처리 완료된 파일 저장
daily_df.to_csv("어린이대공원_일일입장객_최종.csv", index=False, encoding="utf-8-sig")

FileNotFoundError: [Errno 2] No such file or directory: '서울시설공단_어린이대공원 입장객 인원 정보_20260714.csv'

In [ ]:
import pandas as pd

# 1. 방금 저장한 전처리 데이터 불러오기
df = pd.read_csv("어린이대공원_일일입장객_최종.csv")
df['입장일'] = pd.to_datetime(df['입장일'])

# 2. 과거 시계열 데이터(힌트) 생성하기 (shift 함수 활용)
df['전날_방문객'] = df['총입장객수'].shift(1)
df['전전날_방문객'] = df['총입장객수'].shift(2)
# 365일 전 데이터를 작년 동기 데이터로 설정
df['작년동기_방문객'] = df['총입장객수'].shift(365)

# 3. 결측치가 생긴 앞부분 처리
# (과거 데이터가 없는 첫 1년은 결측치(NaN)가 되므로 모델 학습을 위해 제거)
df_model = df.dropna().reset_index(drop=True)

print(df_model.head())
print(f"\n최종 모델 학습용 데이터 개수: {len(df_model)}일")

In [ ]:
# 한국 공휴일을 계산하기 위한 라이브러리 설치 (코랩 환경)
!pip install holidays

import pandas as pd
import holidays

# 1. "가장 처음 업로드하셨던 원본 데이터" 다시 불러오기
df_raw = pd.read_csv("서울시설공단_어린이대공원 입장객 인원 정보_20260714.csv", encoding="utf-8-sig")
df_raw['입장일'] = pd.to_datetime(df_raw['입장일'])

# 2. 작년 데이터(shift 365)를 구하기 위해 1년 전인 '2022년 2월 1일'부터 넉넉히 필터링
mask = (df_raw['입장일'] >= '2022-02-01') & (df_raw['입장일'] <= '2025-01-31')
df_filtered = df_raw.loc[mask]

# 3. 일별 그룹화 및 총입장객수 합산
df = df_filtered.groupby('입장일', as_index=False)['입장인원'].sum()
df.rename(columns={'입장인원': '총입장객수'}, inplace=True)

# 4. 한국 공휴일 정보 가져오기 (2022년 ~ 2025년)
kr_holidays = holidays.KR(years=[2022, 2023, 2024, 2025])

def get_holiday_type(date):
    if date in kr_holidays:
        return 2  # 법정 공휴일
    elif date.dayofweek >= 5:
        return 1  # 주말
    else:
        return 0  # 평일

# 5. 휴일유형 추가
df['휴일유형'] = df['입장일'].apply(get_holiday_type)

# 6. 과거 시계열 데이터(어제, 엊그제, 작년) 생성
df['전날_방문객'] = df['총입장객수'].shift(1)
df['전전날_방문객'] = df['총입장객수'].shift(2)
df['작년동기_방문객'] = df['총입장객수'].shift(365)

# 7. ★핵심: 2022년(과거 데이터를 만들기 위한 더미 데이터)을 잘라내고 목표했던 2023-02-01부터 남기기
df_model = df[df['입장일'] >= '2023-02-01'].reset_index(drop=True)

# 8. 결측치 확인 및 결과 출력
print("📊 2023년 2월 1일부터 시작하는지 확인:")
display(df_model.head())
print(f"\n최종 모델 학습용 데이터 개수: {len(df_model)}일")

# 9. 수정된 파일 저장
df_model.to_csv("어린이대공원_모델입력_1차_수정.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

# 1. 앞서 완성한 1차 뼈대 데이터 불러오기
df_base = pd.read_csv("어린이대공원_모델입력_1차_수정.csv", encoding="utf-8-sig")
df_base['입장일'] = pd.to_datetime(df_base['입장일'])

# 2. 기상청 서울 날씨 데이터 불러오기
# (기상청 CSV 파일은 보통 cp949 인코딩을 사용합니다)
try:
    df_weather = pd.read_csv("서울날씨.csv", encoding="cp949")
except:
    df_weather = pd.read_csv("서울날씨.csv", encoding="utf-8")

# 기상청 날짜 컬럼('일시')을 datetime으로 변환
df_weather['일시'] = pd.to_datetime(df_weather['일시'])

# 3. 데이터 결합 (Left Join)
# '입장일'과 기상청의 '일시'를 기준으로 1:1 결합합니다.
df_merged = pd.merge(df_base, df_weather, left_on='입장일', right_on='일시', how='left')

# 4. 불필요한 컬럼 정리 및 결측치(NaN) 대치
# 결합 후 중복되는 기상청 '일시', '지점', '지점명' 컬럼은 삭제
df_merged = df_merged.drop(columns=['지점', '지점명', '일시'], errors='ignore')

# ★핵심: 기상청 데이터는 비나 눈이 안 온 날은 강수량이 0이 아니라 빈칸(NaN)으로 비워져 있습니다.
# 모델이 에러를 뿜지 않도록 강수량/적설량의 빈칸을 모두 0으로 채워줍니다.
cols_to_fill_zero = [col for col in df_merged.columns if '강수' in col or '적설' in col]
df_merged[cols_to_fill_zero] = df_merged[cols_to_fill_zero].fillna(0)

# 5. 결과 확인
print("✨ 날씨 데이터 병합 완료! 최종 데이터 미리보기:")
display(df_merged.head())
print(f"\n최종 컬럼 목록: {df_merged.columns.tolist()}")

# 6. 2차 파일로 중간 저장
df_merged.to_csv("어린이대공원_모델입력_2차.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

# 1. 2차 데이터(방문객 + 날씨) 불러오기
df_base = pd.read_csv("어린이대공원_모델입력_2차.csv", encoding="utf-8-sig")
df_base['입장일'] = pd.to_datetime(df_base['입장일'])

# 2. 네이버 데이터랩 검색량(엑셀) 불러오기
# ★핵심: 네이버 데이터랩 엑셀 파일은 위 6줄에 설명이 있으므로 건너뛰고(skiprows=6) 읽어옵니다.
df_search = pd.read_excel("검색량datalab.xlsx", skiprows=6)

# 컬럼명을 알아보기 쉽게 변경하고 날짜 타입 맞추기
df_search.columns = ['날짜', '검색량']
df_search['날짜'] = pd.to_datetime(df_search['날짜'])

# 3. 데이터 결합 (Left Join)
df_final = pd.merge(df_base, df_search, left_on='입장일', right_on='날짜', how='left')

# 4. 불필요한 날짜 컬럼 제거 및 결측치 처리 (누락된 검색량을 0으로 대치)
df_final = df_final.drop(columns=['날짜'])
df_final['검색량'] = df_final['검색량'].fillna(0)

# 5. 완성된 최종 데이터 확인
print("🎉 모든 입력 노드 병합 완료! 마스터 데이터 미리보기:")
display(df_final.head())
print(f"\n최종 컬럼 목록: {df_final.columns.tolist()}")

# 6. 최종 마스터 파일 저장
df_final.to_csv("어린이대공원_GA_ACO_BP_마스터데이터.csv", index=False, encoding="utf-8-sig")

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


🎉 모든 입력 노드 병합 완료! 마스터 데이터 미리보기:


,입장일,총입장객수,휴일유형,전날_방문객,전전날_방문객,작년동기_방문객,평균기온(°C),최저기온(°C),최고기온(°C),일강수량(mm),평균 풍속(m/s),일 최심신적설(cm),검색량
0,2023-02-01,12149,0,12409.0,10667.0,NaN,0.9,-3.0,6.0,0.0,3.2,0.0,3.06664
1,2023-02-02,11413,0,12149.0,12409.0,10150.0,-2.4,-5.1,1.4,0.0,2.5,0.0,3.24489
2,2023-02-03,10944,0,11413.0,12149.0,13449.0,-1.6,-3.9,3.4,0.0,2.2,0.0,4.05828
3,2023-02-04,19197,1,10944.0,11413.0,10781.0,-0.2,-5.2,6.0,0.0,2.0,0.0,6.40499
4,2023-02-05,21548,1,19197.0,10944.0,10089.0,1.7,-3.3,7.1,0.0,1.8,0.0,6.14367



최종 컬럼 목록: ['입장일', '총입장객수', '휴일유형', '전날_방문객', '전전날_방문객', '작년동기_방문객', '평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '일강수량(mm)', '평균 풍속(m/s)', '일 최심신적설(cm)', '검색량']


In [ ]:
from google.colab import drive

# 구글 드라이브 마운트 (연결)
drive.mount('/content/drive')

# 작업 경로를 '정선군 프로젝트' 폴더로 이동
import os
os.chdir('/content/drive/MyDrive/정선군 프로젝트')

print("✅ 구글 드라이브 연결 및 작업 경로 이동 완료!")
# 현재 폴더 안에 있는 파일/폴더 목록 확인
print(os.listdir('.'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 구글 드라이브 연결 및 작업 경로 이동 완료!
['01_데이터전처리.ipynb', 'data', '02_GA_ACO_BP_모델링.ipynb', 'models', 'docs']


In [ ]:
import pandas as pd
import numpy as np

print("🔄 데이터 피처 엔지니어링 시작...")

# 1. 원본 마스터 데이터 불러오기
file_path = "data/어린이대공원_GA_ACO_BP_마스터데이터.csv" # 경로 확인
df = pd.read_csv(file_path, encoding="utf-8-sig")

# '입장일'을 날짜 타입(datetime)으로 변환하여 날짜 비교가 가능하게 설정
df['입장일'] = pd.to_datetime(df['입장일'])


# --- 2. 파생 변수(Feature) 추가 ---

# ① 우천 여부 이진화 (강수량이 0을 초과하면 1, 아니면 0)
# (주의: 원본 컬럼명이 '일강수량(mm)'이 맞는지 확인하세요)
df['비가왔는가'] = (df['일강수량(mm)'] > 0.0).astype(int)

# ② 폭염 여부 이진화 (최고기온이 30도 이상이면 1, 아니면 0)
df['폭염인가'] = (df['최고기온(°C)'] >= 30.0).astype(int)

# ③ 전주 동요일 방문객 (정확히 7일 전의 '총입장객수' 가져오기)
df['전주_동요일_방문객'] = df['총입장객수'].shift(7)
# 앞의 7일은 과거 데이터가 없으므로 결측치(NaN)가 생깁니다. 이를 바로 뒤의 값으로 채워줍니다.
df['전주_동요일_방문객'] = df['전주_동요일_방문객'].bfill()

# ④ 명절 연휴 여부 (설날, 추석 연휴 하드코딩 필터링)
# 데이터 기간(2023~2024년)에 맞춘 주요 명절 연휴 날짜입니다.
# (필요시 본인 데이터 기간에 맞게 날짜를 더 추가/수정하세요)
traditional_holidays = [
    # 2023년 설날 연휴
    '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    # 2023년 추석 연휴
    '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03',
    # 2024년 설날 연휴
    '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    # 2024년 추석 연휴
    '2024-09-14', '2024-09-15', '2024-09-16', '2024-09-17', '2024-09-18'
]
traditional_holidays = pd.to_datetime(traditional_holidays)

# 입장일이 명절 리스트 안에 있으면 1, 아니면 0
df['명절인가'] = df['입장일'].isin(traditional_holidays).astype(int)


# --- 3. 결과 확인 및 저장 ---
# 추가된 변수들이 잘 들어갔는지 콘솔에서 확인 (상위 15개 행)
check_columns = ['입장일', '일강수량(mm)', '비가왔는가', '최고기온(°C)', '폭염인가', '총입장객수', '전주_동요일_방문객', '명절인가']
print("\n👀 [데이터 변환 결과 미리보기]")
print(df[check_columns].head(15))

# 기존 원본을 보존하기 위해 새로운 이름으로 저장
new_file_path = "data/어린이대공원_업그레이드_마스터데이터.csv"
df.to_csv(new_file_path, index=False, encoding="utf-8-sig")

print(f"\n✅ 4개의 새로운 피처가 추가된 데이터가 '{new_file_path}'로 성공적으로 저장되었습니다!")

🔄 데이터 피처 엔지니어링 시작...

👀 [데이터 변환 결과 미리보기]
          입장일  일강수량(mm)  비가왔는가  최고기온(°C)  폭염인가  총입장객수  전주_동요일_방문객  명절인가
0  2023-02-01       0.0      0       6.0     0  12149     12149.0     0
1  2023-02-02       0.0      0       1.4     0  11413     12149.0     0
2  2023-02-03       0.0      0       3.4     0  10944     12149.0     0
3  2023-02-04       0.0      0       6.0     0  19197     12149.0     0
4  2023-02-05       0.0      0       7.1     0  21548     12149.0     0
5  2023-02-06       0.0      0       8.8     0  11738     12149.0     0
6  2023-02-07       0.0      0       9.5     0  13366     12149.0     0
7  2023-02-08       0.0      0       7.8     0  13548     12149.0     0
8  2023-02-09       0.0      0       9.9     0  14648     11413.0     0
9  2023-02-10       0.6      1       6.2     0  12461     10944.0     0
10 2023-02-11       0.0      0       8.5     0  23722     19197.0     0
11 2023-02-12       0.0      0       7.6     0  23207     21548.0     0
12 2023-02-13       0.0

In [ ]:
import pandas as pd
import numpy as np

print("🔄 2차 추가 피처 엔지니어링 (상호작용 및 세부 휴일) 시작...")

# 1. 1차 업그레이드된 데이터 불러오기
# ⚠️ 주의: '경로 복사'를 통해 본인의 구글 드라이브 실제 경로로 반드시 수정해 주세요!
file_path = "data/어린이대공원_업그레이드_마스터데이터.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")
df['입장일'] = pd.to_datetime(df['입장일'])

# --- 2. 강력한 파생 변수(Feature) 3가지 추가 ---

# ① 휴일_우천_페널티 (교호작용 변수)
# 휴일(주말 1, 공휴일 2)이면서 비가 왔을 때(1) 비선형적으로 방문객이 급감하는 패턴을 학습시킵니다.
# 휴일유형 값에 비가왔는가(0 또는 1)를 곱하면 비 오는 휴일만 값이 1 또는 2로 활성화됩니다.
df['휴일_우천_페널티'] = df['휴일유형'] * df['비가왔는가']


# ② 임시공휴일 여부
# 사람들이 미리 계획을 세우기 힘들어 일반 공휴일보다 방문객이 적은 날을 명시합니다.
# 2024년 10월 1일(국군의 날)을 포함하여 23~24년 주요 임시공휴일을 지정합니다.
temp_holidays = pd.to_datetime([
    '2023-10-02', # 추석 연휴와 개천절 사이 임시공휴일
    '2024-10-01'  # 6만 명으로 잘못 예측했던 국군의 날 임시공휴일
])
df['임시공휴일_여부'] = df['입장일'].isin(temp_holidays).astype(int)


# ③ 징검다리 휴일 여부
# 평일이지만 앞뒤로 주말이나 공휴일이 있어서 유독 휴가 쓰는 사람이 많은 날을 명시합니다.
# 오답 노트에 있었던 24년 9월 30일(23일 차)이 대표적인 징검다리 평일입니다.
bridge_holidays = pd.to_datetime([
    '2023-06-05', '2023-08-14',
    '2024-08-16', '2024-09-30'
])
df['징검다리_휴일_여부'] = df['입장일'].isin(bridge_holidays).astype(int)


# --- 3. 변환 결과 확인 및 저장 ---
check_cols = ['입장일', '총입장객수', '휴일유형', '비가왔는가', '휴일_우천_페널티', '임시공휴일_여부', '징검다리_휴일_여부']
print("\n👀 [문제가 되었던 24년 9월 말 ~ 10월 초 데이터 확인]")
# 모델이 헷갈려 했던 구간만 콕 집어서 출력해 봅니다.
display_df = df[(df['입장일'] >= '2024-09-29') & (df['입장일'] <= '2024-10-02')]
print(display_df[check_cols])

# 모델 학습용으로 새로운 파일명으로 저장
# ⚠️ 주의: 저장할 경로도 본인 드라이브 경로로 맞춰주세요.
new_file_path = "data/어린이대공원_최종_마스터데이터.csv"
df.to_csv(new_file_path, index=False, encoding="utf-8-sig")

print(f"\n✅ 3개의 피처가 추가된 최종 데이터가 '{new_file_path}'로 성공적으로 저장되었습니다!")

🔄 2차 추가 피처 엔지니어링 (상호작용 및 세부 휴일) 시작...

👀 [문제가 되었던 24년 9월 말 ~ 10월 초 데이터 확인]
           입장일  총입장객수  휴일유형  비가왔는가  휴일_우천_페널티  임시공휴일_여부  징검다리_휴일_여부
606 2024-09-29  36325     1      0          0         0           0
607 2024-09-30  17477     0      0          0         0           1
608 2024-10-01  23637     2      1          2         1           0
609 2024-10-02  19333     0      0          0         0           0

✅ 3개의 피처가 추가된 최종 데이터가 'data/어린이대공원_최종_마스터데이터.csv'로 성공적으로 저장되었습니다!
